# 二阶段绝缘子分类训练

这个 notebook 直接读取已经生成好的分类数据集，并启动 Ultralytics 分类训练。


## 0. 依赖导入


In [ ]:
from pathlib import Path
import json
from datetime import datetime

from ultralytics import YOLO

print('✓ 依赖导入完成')


## 1. 核心函数


In [ ]:
def ensure_classification_dataset(dataset_dir):
    dataset_dir = Path(dataset_dir)
    required_dirs = [dataset_dir / 'train', dataset_dir / 'val']

    for directory in required_dirs:
        if not directory.exists():
            raise ValueError(f'Missing dataset split directory: {directory}')
        class_dirs = [path for path in directory.iterdir() if path.is_dir()]
        if not class_dirs:
            raise ValueError(f'No class directories found in: {directory}')

    return dataset_dir


def build_training_config(
    dataset_dir,
    experiment_root,
    experiment_name,
    model_weights='yolo11n-cls.pt',
    epochs=100,
    imgsz=224,
    batch=32,
    device='0',
):
    return {
        'model': model_weights,
        'data': str(Path(dataset_dir)),
        'project': str(Path(experiment_root)),
        'name': experiment_name,
        'epochs': epochs,
        'imgsz': imgsz,
        'batch': batch,
        'device': device,
        'save': True,
        'exist_ok': True,
        'amp': False,
    }


def write_experiment_metadata(output_dir, config, dataset_summary=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    metadata = {
        'created_at': datetime.now().isoformat(),
        'config': config,
        'dataset_summary': dataset_summary or {},
    }
    metadata_path = output_dir / 'classifier_experiment.json'
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
    return metadata_path


def run_training(config):
    model = YOLO(config['model'])
    return model.train(**{key: value for key, value in config.items() if key != 'model'})


print('✓ 核心函数准备完成')


## 2. 路径与参数配置


In [ ]:
DATASET_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets/insulator_cls_dataset_tight')
EXPERIMENT_ROOT = Path('/home/hujing/yolo-web-ui/experiments_cls')
EXPERIMENT_NAME = 'insulator_cls_exp001'

MODEL_WEIGHTS = 'yolo11n-cls.pt'
EPOCHS = 100
IMGSZ = 224
BATCH = 32
DEVICE = '0'

print('DATASET_DIR =', DATASET_DIR)
print('EXPERIMENT_ROOT =', EXPERIMENT_ROOT)
print('EXPERIMENT_NAME =', EXPERIMENT_NAME)


## 3. 检查数据集


In [ ]:
validated_dataset_dir = ensure_classification_dataset(DATASET_DIR)
print('✓ 数据集目录检查通过:', validated_dataset_dir)

summary_path = DATASET_DIR / 'dataset_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))
else:
    summary = {}
    print('未找到 dataset_summary.json')


## 4. 生成训练配置


In [ ]:
config = build_training_config(
    dataset_dir=validated_dataset_dir,
    experiment_root=EXPERIMENT_ROOT,
    experiment_name=EXPERIMENT_NAME,
    model_weights=MODEL_WEIGHTS,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
)

print(json.dumps(config, indent=2, ensure_ascii=False))


## 5. 写入实验 metadata


In [ ]:
metadata_path = write_experiment_metadata(
    output_dir=EXPERIMENT_ROOT / EXPERIMENT_NAME,
    config=config,
    dataset_summary=summary,
)
print('metadata_path =', metadata_path)


## 6. 启动训练


In [ ]:
results = run_training(config)
results


## 7. 输出目录提示


In [ ]:
print('训练输出目录:')
print(EXPERIMENT_ROOT / EXPERIMENT_NAME)
